In [ ]:
# In[1]:

import pandas as pd

# Load the metric_container.csv file
metric_container_path = 'dataset/Bank/telemetry/2021_03_09/metric/metric_container.csv'
metric_container_df = pd.read_csv(metric_container_path)

# Extract unique KPI names and their corresponding components
kpi_component_mapping = metric_container_df[['cmdb_id', 'kpi_name']].drop_duplicates()

kpi_component_mapping

```
Out[1]:
```
The `metric_container.csv` file contains 1,689 unique combinations of KPI names and their corresponding components. These include various KPIs such as CPU utilization, process metrics, session metrics, and thread metrics, associated with components like Tomcat04, MG01, and Tomcat02, among others. For a detailed view of specific KPIs and components, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name
0       Tomcat04                         OSLinux-CPU_CPU_CPUCpuUtil
1           MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies
2           MG01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem
3           MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount
4           MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
...          ...                                                ...
520542  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...
520543  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
520544  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
520545  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...
520547  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...

[1689 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Define the possible root cause components
root_cause_components = [
    'apache01', 'apache02', 'Tomcat01', 'Tomcat02', 'Tomcat04', 'Tomcat03',
    'MG01', 'MG02', 'IG01', 'IG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02'
]

# Filter the data to include only the KPIs associated with the root cause components
filtered_metric_container_df = metric_container_df[metric_container_df['cmdb_id'].isin(root_cause_components)]

# Create a new column to classify time series by 'component-KPI'
filtered_metric_container_df['component-KPI'] = filtered_metric_container_df['cmdb_id'] + '-' + filtered_metric_container_df['kpi_name']

# Aggregate the data to create time series classified by 'component-KPI'
time_series_df = filtered_metric_container_df[['timestamp', 'component-KPI', 'value']]

time_series_df

```
Out[2]:
```
The filtered data includes 1,221,440 rows of time series information for KPIs associated with the possible root cause components. Each row represents a timestamp, a 'component-KPI' classification (e.g., Tomcat04-OSLinux-CPU_CPU_CPUCpuUtil), and the corresponding KPI value. This dataset can be used for further analysis of component-specific KPI trends over time.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                                      component-KPI    value
0        1615219200                Tomcat04-OSLinux-CPU_CPU_CPUCpuUtil  26.1244
1        1615219200  MG01-OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZom...   2.0000
2        1615219200     MG01-OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  58.0000
3        1615219200    MG01-OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount   3.0000
4        1615219200  MG01-OSLinux-OSLinux_PROCESS_zabbix-zabbix_age...   0.3000
...             ...                                                ...      ...
1226459  1615305540  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...   0.8300
1226460  1615305540  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...   6.0000
1226461  1615305540  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...   0.4000
1226462  1615305540                   MG01-OSLinux-CPU_CPU_CPUidleutil  73.3278
1226463  1615305540              Redis02-redis-Redis_6379_redis server   1.0000

[1221440 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = time_series_df.groupby('component-KPI')['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'P95_threshold'}, inplace=True)

p95_thresholds

```
Out[3]:
```
The global P95 thresholds have been calculated for each of the 1,651 unique 'component-KPI' time series in the dataset. These thresholds represent the 95th percentile of the KPI values across the entire dataset and can be used to identify anomalies during specific failure durations. For a detailed view of specific thresholds, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

component-KPI  P95_threshold
0         IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryMax   2.040136e+09
1       IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsage   4.139625e+01
2        IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsed   8.476656e+08
3      IG01-JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed   5.795390e+07
4        IG01-JVM-Operating System_7778_JVM_JVM_CPULoad   5.109200e-01
...                                                 ...            ...
1646              apache02-OSLinux-OSLinux_SWAP_SWAP_Si   0.000000e+00
1647              apache02-OSLinux-OSLinux_SWAP_SWAP_So   0.000000e+00
1648  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-D...   1.000000e+00
1649  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-H...   1.000000e+00
1650        apache02-OSLinux-OSLinux_ZABBIX_Host_Uptime   6.499915e+06

[1651 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the failure duration in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_time = int(datetime(2021, 3, 9, 9, 0, tzinfo=tz).timestamp())
end_time = int(datetime(2021, 3, 9, 9, 30, tzinfo=tz).timestamp())

# Filter the time series data to include only rows within the failure duration
filtered_time_series_df = time_series_df[
    (time_series_df['timestamp'] >= start_time) & (time_series_df['timestamp'] <= end_time)
]

filtered_time_series_df

```
Out[4]:
```
The filtered 'component-KPI' time series data includes 26,038 rows where the timestamps fall within the failure duration of March 9, 2021, from 09:00 to 09:30 (UTC+8). Each row represents a specific 'component-KPI' and its corresponding value during this time period. This dataset is ready for further anomaly detection or analysis.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                                      component-KPI        value
450893  1615251240  apache01-OSLinux-OSLinux_MEMORY_MEMORY_NoCache...    25.381700
450894  1615251240  apache01-OSLinux-OSLinux_MEMORY_MEMORY_MEMUsed...    47.000000
450895  1615251240  apache01-OSLinux-OSLinux_FILESYSTEM_-tmp_FSCap...    12.640700
450896  1615251240     apache01-OSLinux-OSLinux_MEMORY_MEMORY_UserMem  1502.000000
450897  1615251240  apache01-OSLinux-OSLinux_NETWORK_NETWORK_TCP-C...     0.000000
...            ...                                                ...          ...
477029  1615253040  apache02-OSLinux-OSLinux_FILESYSTEM_-apache_FS...     8.917379
477030  1615253040  apache02-OSLinux-OSLinux_FILESYSTEM_-_FSInodeU...    29.614054
477031  1615253040   apache02-OSLinux-OSLinux_FILESYSTEM_-_FSCapacity    48.975464
477032  1615253040           apache02-OSLinux-CPU_CPU-3_SingleCpuidle    99.302210
477033  1615253040  apache02-OSLinux-OSLinux_FILESYSTEM_-home_FSCa...     2.617405

[26038 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Merge the filtered time series data with the P95 thresholds to identify anomalies
anomalies_df = filtered_time_series_df.merge(p95_thresholds, on='component-KPI', how='left')

# Mark data points as anomalous if their values exceed the P95 threshold
anomalies_df['is_anomalous'] = anomalies_df['value'] > anomalies_df['P95_threshold']

anomalies_df

```
Out[5]:
```
The analysis identified anomalies in the filtered 'component-KPI' time series data by comparing each KPI value against its respective global P95 threshold. Each data point is marked as anomalous (`True`) if its value exceeds the threshold. The dataset contains 26,038 rows, with an additional column indicating whether each data point is anomalous. This information can now be used to investigate potential issues during the failure duration.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                                      component-KPI        value  P95_threshold  is_anomalous
0      1615251240  apache01-OSLinux-OSLinux_MEMORY_MEMORY_NoCache...    25.381700      27.989800         False
1      1615251240  apache01-OSLinux-OSLinux_MEMORY_MEMORY_MEMUsed...    47.000000      53.000000         False
2      1615251240  apache01-OSLinux-OSLinux_FILESYSTEM_-tmp_FSCap...    12.640700      12.641098         False
3      1615251240     apache01-OSLinux-OSLinux_MEMORY_MEMORY_UserMem  1502.000000    1677.000000         False
4      1615251240  apache01-OSLinux-OSLinux_NETWORK_NETWORK_TCP-C...     0.000000       0.000000         False
...           ...                                                ...          ...            ...           ...
26033  1615253040  apache02-OSLinux-OSLinux_FILESYSTEM_-apache_FS...     8.917379       9.089811         False
26034  1615253040  apache02-OSLinux-OSLinux_FILESYSTEM_-_FSInodeU...    29.614054      29.614100         False
26035  1615253040   apache02-OSLinux-OSLinux_FILESYSTEM_-_FSCapacity    48.975464      48.983758         False
26036  1615253040           apache02-OSLinux-CPU_CPU-3_SingleCpuidle    99.302210      99.833887         False
26037  1615253040  apache02-OSLinux-OSLinux_FILESYSTEM_-home_FSCa...     2.617405       2.623900         False

[26038 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Identify faults by grouping consecutive anomalies for each 'component-KPI' time series
anomalies_df['anomaly_group'] = (
    anomalies_df['is_anomalous']
    .groupby([anomalies_df['component-KPI'], (anomalies_df['is_anomalous'] != anomalies_df['is_anomalous'].shift()).cumsum()])
    .transform('cumsum')
)

# Filter out isolated noise spikes (groups with only one anomaly)
grouped_anomalies = anomalies_df[anomalies_df['is_anomalous']].groupby(['component-KPI', 'anomaly_group'])
faults = []

for (component_kpi, group_id), group in grouped_anomalies:
    if len(group) > 1:  # Exclude isolated spikes
        extremal_value = group['value'].max()
        threshold_breach = (group['value'] - group['P95_threshold']).max()
        if threshold_breach > 0.5 * extremal_value:  # Exclude false positives
            faults.append({
                'component-KPI': component_kpi,
                'start_time': group['timestamp'].min(),
                'end_time': group['timestamp'].max(),
                'extremal_value': extremal_value,
                'threshold_breach': threshold_breach
            })

# Convert faults to a DataFrame for better readability
faults_df = pd.DataFrame(faults)

faults_df

```
Out[6]:
```
The analysis identified faults by grouping consecutive anomalies for each 'component-KPI' time series. Six faults were detected, each characterized by a start time, end time, extremal value, and threshold breach. These faults represent sustained periods of abnormal behavior, excluding isolated noise spikes and false positives where the threshold breach was less than or equal to 50% of the extremal value. This refined dataset can be used for further investigation into system issues.

The original code execution output of IPython Kernel is also provided below for reference:

component-KPI  start_time    end_time  extremal_value  threshold_breach
0               MG01-OSLinux-CPU_CPU-0_SingleCpuUtil  1615251960  1615252920        100.0000         93.495400
1  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...  1615252200  1615252440          0.7000          0.500000
2  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...  1615252200  1615252800          0.1833          0.142485
3  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...  1615252200  1615252800          6.0000          5.360015
4  Redis02-OSLinux-OSLinux_NETWORK_NETWORK_TCP-FI...  1615252560  1615252860          1.0000          1.000000
5  Tomcat02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...  1615251720  1615252800          6.0000          5.000000```
```

In [ ]:
# In[7]:

# Identify the fault with the most significant deviation from its threshold
most_significant_fault = faults_df.loc[faults_df['threshold_breach'].idxmax()]

# Extract the component, KPI, and occurrence time of this fault
most_significant_fault_summary = {
    'component-KPI': most_significant_fault['component-KPI'],
    'start_time': most_significant_fault['start_time'],
    'end_time': most_significant_fault['end_time'],
    'threshold_breach': most_significant_fault['threshold_breach']
}

most_significant_fault_summary

```
Out[7]:
```
The fault with the most significant deviation from its threshold occurred in the `MG01` component for the KPI `OSLinux-CPU_CPU-0_SingleCpuUtil`. This fault lasted from 09:06 to 09:22 (UTC+8) on March 9, 2021, with a threshold breach of 93.4954, indicating a severe anomaly in CPU utilization. This is likely the root cause of the issue during the failure duration.

The original code execution output of IPython Kernel is also provided below for reference:

{'component-KPI': 'MG01-OSLinux-CPU_CPU-0_SingleCpuUtil', 'start_time': 1615251960, 'end_time': 1615252920, 'threshold_breach': 93.4954}```
```

In [ ]:
# In[8]:

# Extract the first anomalous data point within the fault for the 'MG01-OSLinux-CPU_CPU-0_SingleCpuUtil' time series
fault_series = anomalies_df[
    (anomalies_df['component-KPI'] == 'MG01-OSLinux-CPU_CPU-0_SingleCpuUtil') &
    (anomalies_df['timestamp'] >= most_significant_fault['start_time']) &
    (anomalies_df['timestamp'] <= most_significant_fault['end_time']) &
    (anomalies_df['is_anomalous'])
]

# Get the timestamp of the first anomalous data point
root_cause_timestamp = fault_series['timestamp'].min()

# Convert the timestamp to a human-readable datetime in UTC+8
root_cause_datetime = datetime.fromtimestamp(root_cause_timestamp, tz=tz)

root_cause_datetime

```
Out[8]:
```
The root cause occurrence datetime for the fault in the `MG01-OSLinux-CPU_CPU-0_SingleCpuUtil` time series is March 9, 2021, at 09:06 (UTC+8). This marks the first anomalous data point within the identified fault.

The original code execution output of IPython Kernel is also provided below for reference:

2021-03-09 09:06:00+08:00```
```